In [ ]:
# ----------------------------------------------------------
# Setup
# ----------------------------------------------------------
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("pandas version:", pd.__version__)

pandas version: 2.2.3


In [ ]:
# ----------------------------------------------------------
# Task 1.1 | Load the dataset
# ----------------------------------------------------------
CANDIDATE_PATHS = [
    "Railway_info.csv",
    "data/raw/Railway_info.csv",
    "trains.csv",
    "data/raw/trains.csv",
    "data/raw/sample_trains.csv",
]

def load_dataset():
    """Return (dataframe, source_path). Falls back to a Colab upload widget."""
    for path in CANDIDATE_PATHS:
        if os.path.exists(path):
            print(f"✅ Loading dataset from: {path}")
            return pd.read_csv(path), path
    try:  # Google Colab upload fallback
        from google.colab import files
        print("Dataset not found in this session — please upload your CSV:")
        uploaded = files.upload()
        name = next(iter(uploaded))
        return pd.read_csv(name), name
    except ImportError:
        raise FileNotFoundError(
            "Could not locate the dataset CSV. Upload it (Colab: folder icon) and re-run."
        )

df, source_path = load_dataset()
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# First 10 rows, as required
df.head(10)

✅ Loading dataset from: Railway_info.csv
Rows: 11,113 | Columns: 5


,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days
0,107,SWV-MAO-VLNK,SAWANTWADI ROAD,MADGOAN JN.,Saturday
1,108,VLNK-MAO-SWV,MADGOAN JN.,SAWANTWADI ROAD,Friday
2,128,MAO-KOP SPEC,MADGOAN JN.,CHHATRAPATI SHAHU MAHARAJ TERMINUS,Friday
3,290,PALACE ON WH,DELHI-SAFDAR JANG,DELHI-SAFDAR JANG,Wednesday
4,401,BSB BHARATDA,AURANGABAD,VARANASI JN.,Saturday
5,421,LKO-SVDK FTR,LUCKNOW JN.,SHRI MATA VAISHNO DEVI KATRA,Tuesday
6,422,SVDK-LKO FTR,SHRI MATA VAISHNO DEVI KATRA,LUCKNOW JN.,Monday
7,477,FTR TRAIN NO,SIRSA,SIRSA,Sunday
8,502,RJPB-UMB FTR,RAJENDRANAGAR TERMINAL,AMBALA CANTT JN,Monday
9,504,PNBE-BTI FTR,PATNA JN.,BATHINDA JN,Wednesday


In [ ]:
# ----------------------------------------------------------
# Task 1.1 | Structure, data types & missing values
# ----------------------------------------------------------
df.info()

missing = df.isnull().sum().rename("missing_values").to_frame()
missing["missing_pct"] = (missing["missing_values"] / len(df) * 100).round(2)
print("\nMissing-value report:")
print(missing)

df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11113 entries, 0 to 11112
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Train_No                  11113 non-null  int64 
 1   Train_Name                11113 non-null  object
 2   Source_Station_Name       11113 non-null  object
 3   Destination_Station_Name  11113 non-null  object
 4   days                      11113 non-null  object
dtypes: int64(1), object(4)
memory usage: 434.2+ KB

Missing-value report:
                          missing_values  missing_pct
Train_No                               0          0.0
Train_Name                             0          0.0
Source_Station_Name                    0          0.0
Destination_Station_Name               0          0.0
days                                   0          0.0


,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days
count,11113.000000,11113,11113,11113,11113
unique,NaN,7580,921,924,14
top,NaN,TBM-MSB EMU,CST-MUMBAI,CST-MUMBAI,Friday
freq,NaN,139,513,514,1471
mean,49190.570413,NaN,NaN,NaN,NaN
std,28515.986645,NaN,NaN,NaN,NaN
min,107.000000,NaN,NaN,NaN,NaN
25%,22607.000000,NaN,NaN,NaN,NaN
50%,47174.000000,NaN,NaN,NaN,NaN
75%,68012.000000,NaN,NaN,NaN,NaN


In [ ]:
# ----------------------------------------------------------
# Task 1.2 | Resolve key columns (robust to header naming)
# ----------------------------------------------------------
def find_column(frame, *patterns):
    """First column whose lower-cased name contains any of the patterns."""
    for pattern in patterns:
        for col in frame.columns:
            if pattern in col.lower():
                return col
    return None

train_col  = find_column(df, "train no", "train_no", "train id", "train code", "train") or df.columns[0]
source_col = find_column(df, "source", "origin") or find_column(df, "from")
dest_col   = find_column(df, "destination") or find_column(df, "to")

print(f"Train column       : {train_col}")
print(f"Source column      : {source_col}")
print(f"Destination column : {dest_col}")

Train column       : Train_No
Source column      : Source_Station_Name
Destination column : Destination_Station_Name


In [ ]:
# ----------------------------------------------------------
# Task 1.2 | Counts & most common stations
# ----------------------------------------------------------
n_trains  = df[train_col].nunique()
n_sources = df[source_col].nunique()
n_dests   = df[dest_col].nunique()

print(f"Number of trains             : {n_trains:,}")
print(f"Unique source stations       : {n_sources:,}")
print(f"Unique destination stations  : {n_dests:,}")

src_counts = df[source_col].value_counts()
dst_counts = df[dest_col].value_counts()

print(f"\nMost common source station      : {src_counts.idxmax()} ({src_counts.max():,} departures)")
print(f"Most common destination station : {dst_counts.idxmax()} ({dst_counts.max():,} arrivals)")

print("\nTop 5 source stations:")
print(src_counts.head())
print("\nTop 5 destination stations:")
print(dst_counts.head())

Number of trains             : 11,113
Unique source stations       : 921
Unique destination stations  : 924

Most common source station      : CST-MUMBAI (513 departures)
Most common destination station : CST-MUMBAI (514 arrivals)

Top 5 source stations:
Source_Station_Name
CST-MUMBAI       513
SEALDAH          372
CHENNAI BEACH    339
HOWRAH JN.       338
KALYAN JN        285
Name: count, dtype: int64

Top 5 destination stations:
Destination_Station_Name
CST-MUMBAI       514
SEALDAH          373
CHENNAI BEACH    342
HOWRAH JN.       337
KALYAN JN        284
Name: count, dtype: int64


In [ ]:
# ----------------------------------------------------------
# Task 1.3 | Identify & handle missing values
# ----------------------------------------------------------
rows_before = len(df)
print("Missing BEFORE cleaning:")
miss_before = df.isnull().sum()
miss_before = miss_before[miss_before > 0]
print(miss_before if not miss_before.empty else "   (none — no missing values detected ✅)")

# Cleaning policy:
#  1) drop rows that carry no information at all
#  2) a train record without an identifier is unusable -> drop
#  3) missing station names -> explicit 'UNKNOWN' placeholder
#  4) numeric measure columns -> median imputation (identifiers are NEVER imputed)
#  5) any other string column (e.g. times) -> 'UNKNOWN' placeholder
df = df.dropna(how="all")
df = df.dropna(subset=[train_col])
df[train_col] = df[train_col].astype("Int64")   # clean integer ids

df[source_col] = df[source_col].fillna("UNKNOWN")
df[dest_col]   = df[dest_col].fillna("UNKNOWN")

for col in df.select_dtypes(include=[np.number]).columns:
    if col != train_col and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object"]).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna("UNKNOWN")

print("\nMissing AFTER cleaning:", df.isnull().sum().sum(), "value(s)")
print(f"Rows kept: {len(df):,} / {rows_before:,}")

Missing BEFORE cleaning:
   (none — no missing values detected ✅)

Missing AFTER cleaning: 0 value(s)
Rows kept: 11,113 / 11,113


In [ ]:
# ----------------------------------------------------------
# Task 1.3 | Standardize station names (trim + upper-case)
# ----------------------------------------------------------
def standardize_station(value):
    """Trim, collapse inner spaces and upper-case a station name."""
    if pd.isna(value):
        return value
    return " ".join(str(value).split()).upper()

print("Before:", df[source_col].head(8).tolist())
src_before = df[source_col].nunique()
dst_before = df[dest_col].nunique()

df[source_col] = df[source_col].apply(standardize_station)
df[dest_col]   = df[dest_col].apply(standardize_station)

name_col = find_column(df, "train name")
if name_col:
    df[name_col] = df[name_col].apply(standardize_station)

print("After :", df[source_col].head(8).tolist())
print(f"\nUnique source stations      : {src_before} -> {df[source_col].nunique()} (after trim + upper-case)")
print(f"Unique destination stations : {dst_before} -> {df[dest_col].nunique()} (after trim + upper-case)")

Before: ['SAWANTWADI ROAD', 'MADGOAN JN.', 'MADGOAN JN.', 'DELHI-SAFDAR JANG', 'AURANGABAD', 'LUCKNOW JN.', 'SHRI MATA VAISHNO DEVI KATRA', 'SIRSA']
After : ['SAWANTWADI ROAD', 'MADGOAN JN.', 'MADGOAN JN.', 'DELHI-SAFDAR JANG', 'AURANGABAD', 'LUCKNOW JN.', 'SHRI MATA VAISHNO DEVI KATRA', 'SIRSA']

Unique source stations      : 921 -> 921 (after trim + upper-case)
Unique destination stations : 924 -> 924 (after trim + upper-case)


In [ ]:
# Recreating 'df' due to potential kernel state loss.
# This cell re-loads the dataset and reapplies all previous cleaning and standardization steps.
import os
import pandas as pd
import numpy as np

# --- Re-load the dataset (from Task 1.1) ---
CANDIDATE_PATHS = [
    "Railway_info.csv",
    "data/raw/Railway_info.csv",
    "trains.csv",
    "data/raw/trains.csv",
    "data/raw/sample_trains.csv",
]

def load_dataset():
    """Return (dataframe, source_path). Falls back to a Colab upload widget."""
    for path in CANDIDATE_PATHS:
        if os.path.exists(path):
            print(f"✅ Loading dataset from: {path}")
            return pd.read_csv(path), path
    try:  # Google Colab upload fallback
        from google.colab import files
        print("Dataset not found in this session — please upload your CSV:")
        uploaded = files.upload()
        name = next(iter(uploaded))
        return pd.read_csv(name), name
    except ImportError:
        raise FileNotFoundError(
            "Could not locate the dataset CSV. Upload it (Colab: folder icon) and re-run."
        )

df, source_path = load_dataset()
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# --- Re-resolve key columns (from Task 1.2) ---
def find_column(frame, *patterns):
    """First column whose lower-cased name contains any of the patterns."""
    for pattern in patterns:
        for col in frame.columns:
            if pattern in col.lower():
                return col
    return None

train_col  = find_column(df, "train no", "train_no", "train id", "train code", "train") or df.columns[0]
source_col = find_column(df, "source", "origin") or find_column(df, "from")
dest_col   = find_column(df, "destination") or find_column(df, "to")

# --- Re-apply missing value handling (from Task 1.3) ---
# Cleaning policy:
#  1) drop rows that carry no information at all
#  2) a train record without an identifier is unusable -> drop
#  3) missing station names -> explicit 'UNKNOWN' placeholder
#  4) numeric measure columns -> median imputation (identifiers are NEVER imputed)
#  5) any other string column (e.g. times) -> 'UNKNOWN' placeholder
df = df.dropna(how="all")
df = df.dropna(subset=[train_col])
df[train_col] = df[train_col].astype("Int64")

df[source_col] = df[source_col].fillna("UNKNOWN")
df[dest_col]   = df[dest_col].fillna("UNKNOWN")

for col in df.select_dtypes(include=[np.number]).columns:
    if col != train_col and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object"]).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna("UNKNOWN")

# --- Re-apply standardization (from Task 1.3) ---
def standardize_station(value):
    """Trim, collapse inner spaces and upper-case a station name."""
    if pd.isna(value):
        return value
    return " ".join(str(value).split()).upper()

df[source_col] = df[source_col].apply(standardize_station)
df[dest_col]   = df[dest_col].apply(standardize_station)

name_col = find_column(df, "train name")
if name_col:
    df[name_col] = df[name_col].apply(standardize_station)

print("\nDataFrame 'df' has been recreated and cleaned to allow subsequent cells to run.")


✅ Loading dataset from: Railway_info.csv
Rows: 11,113 | Columns: 5

DataFrame 'df' has been recreated and cleaned to allow subsequent cells to run.


In [ ]:
# ----------------------------------------------------------
# Export the cleaned dataset
# ----------------------------------------------------------
import os
os.makedirs("data/processed", exist_ok=True)
CLEANED_PATH = "data/processed/trains_cleaned.csv"
df.to_csv(CLEANED_PATH, index=False)
print(f"💾 Cleaned dataset saved to '{CLEANED_PATH}' ({len(df):,} rows).")

try:  # auto-download when running in Colab
    from google.colab import files
    files.download(CLEANED_PATH)
except ImportError:
    pass

df.head()

💾 Cleaned dataset saved to 'data/processed/trains_cleaned.csv' (11,113 rows).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days
0,107,SWV-MAO-VLNK,SAWANTWADI ROAD,MADGOAN JN.,Saturday
1,108,VLNK-MAO-SWV,MADGOAN JN.,SAWANTWADI ROAD,Friday
2,128,MAO-KOP SPEC,MADGOAN JN.,CHHATRAPATI SHAHU MAHARAJ TERMINUS,Friday
3,290,PALACE ON WH,DELHI-SAFDAR JANG,DELHI-SAFDAR JANG,Wednesday
4,401,BSB BHARATDA,AURANGABAD,VARANASI JN.,Saturday
